# Week 3: noisy Steane syndromes and soft ML decoding

This notebook demonstrates the restricted 22-hypothesis experiment used in Week 3. It compares hard thresholding with Gaussian maximum-likelihood decoding while deliberately leaving MAP and coset aggregation for future work.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from experiments.run_quantum_single_pauli import simulate_point
from softqec.quantum_decoders import (
    hard_nearest_syndrome_decode,
    restricted_syndrome_table,
    single_pauli_hypotheses,
    soft_ml_syndrome_decode,
)

## The 22 allowed syndrome signatures

The syndrome order is `[X-check responses | Z-check responses]`.

In [ ]:
labels, _, _ = single_pauli_hypotheses()
syndromes = restricted_syndrome_table()

for label, syndrome in zip(labels, syndromes, strict=True):
    syndrome_text = ''.join(map(str, syndrome))
    print(f"{label:>2}: {syndrome_text}")

## One observation where soft reliability helps

Both decoders receive exactly the same six continuous measurements. The hard decoder thresholds first, whereas the soft decoder retains their magnitudes.

The fixed observation below is a reproducible Gaussian-noise realization for which hard thresholding selects `X6`, while soft ML correctly selects the actual error `Y6`.

In [ ]:
actual = labels.index("Y6")

observation = np.array(
    [
        0.34583,
        0.09429,
        -0.91268,
        0.22358,
        -1.33714,
        -1.16091,
    ]
)

hard = hard_nearest_syndrome_decode(observation)
soft = soft_ml_syndrome_decode(observation)

print("observation:", np.round(observation, 3))
print("actual:", labels[actual])
print("hard estimate:", labels[hard])
print("soft estimate:", labels[soft])

## Why the two decisions differ

The thresholded observation retains only the signs of the six measurements. The soft decoder also uses their distances from zero. A value near zero is treated as uncertain, while a large-magnitude value provides stronger evidence.

In [ ]:
thresholded = (observation < 0.0).astype(np.uint8)

print("thresholded syndrome:", thresholded)
print("actual Y6 syndrome:   ", syndromes[actual])
print("hard X6 syndrome:     ", syndromes[labels.index("X6")])

## Small reproducible comparison

The committed experiment uses 30,000 trials per noise point. This notebook uses a smaller run for interactive inspection.

In [ ]:
sigma_values = [0.2, 0.5, 0.75, 1.0, 1.5, 2.0]

rows = [
    simulate_point(
        sigma_m=sigma,
        trials=5000,
        seed=100 + index,
    )
    for index, sigma in enumerate(sigma_values)
]

hard_rates = [row["hard_rate"] for row in rows]
soft_rates = [row["soft_rate"] for row in rows]

plt.figure(figsize=(7.2, 4.8))

plt.plot(
    sigma_values,
    hard_rates,
    "o-",
    linewidth=2,
    label="Hard threshold + nearest syndrome",
)

plt.plot(
    sigma_values,
    soft_rates,
    "s-",
    linewidth=2,
    label="Soft Gaussian ML",
)

plt.xlabel(r"Syndrome measurement noise $\sigma_m$")
plt.ylabel("Decoding failure rate")
plt.title("Steane [[7,1,3]] single-Pauli decoding")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Interpretation and limits

At small measurement noise, both decoders recover almost every ideal syndrome. At intermediate noise, soft ML performs better because it retains the measurement magnitudes discarded by thresholding. At high noise, both methods become unreliable.

This notebook uses only the identity and single-qubit Pauli errors with equal priors. It does not implement multi-qubit noise, nonuniform priors, MAP decoding, or stabilizer-coset aggregation.